In [31]:
# 3_data_collector.py
import requests
import pandas as pd
import numpy as np

class FPLDataCollector:
    def __init__(self):
        self.base_url = "https://fantasy.premierleague.com/api/"
    
    def get_all_player_data(self):
        """Get current season player data"""
        url = f"{self.base_url}bootstrap-static/"
        response = requests.get(url)
        data = response.json()
        
        # Extract data
        players = pd.DataFrame(data['elements'])
        teams = pd.DataFrame(data['teams'])
        positions = pd.DataFrame(data['element_types'])
        
        # Add team names to players
        players = players.merge(
            teams[['id', 'name', 'short_name']], 
            left_on='team', 
            right_on='id', 
            suffixes=('', '_team')
        )
        
        # Add position names
        position_map = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
        players['position'] = players['element_type'].map(position_map)
        
        # Convert price
        players['price'] = players['now_cost'] / 10
        
        return players, teams, positions
    
    def get_player_history(self, player_id):
        """Get historical data for specific player"""
        url = f"{self.base_url}element-summary/{player_id}/"
        response = requests.get(url)
        data = response.json()
        
        # Get history
        if 'history' in data:
            history = pd.DataFrame(data['history'])
            return history
        return pd.DataFrame()

# Initialize collector
collector = FPLDataCollector()
print("✅ Data collector initialized")

✅ Data collector initialized


In [33]:
# 4_collect_current_data.py
# Get current player data
players, teams, positions = collector.get_all_player_data()

# Save to CSV
players.to_csv('Data/current_players.csv', index=False)
teams.to_csv('Data/teams.csv', index=False)
positions.to_csv('Data/positions.csv', index=False)

print(f"✅ Saved {len(players)} players")
print(f"✅ Saved {len(teams)} teams")
print(f"✅ Saved {len(positions)} positions")

# Show top players
print("\n📊 Top 10 Players by Points:")
top_10 = players.nlargest(10, 'total_points')[['web_name', 'position', 'team', 'price', 'total_points']]
print(top_10)

✅ Saved 755 players
✅ Saved 20 teams
✅ Saved 4 positions

📊 Top 10 Players by Points:
     web_name position  team  price  total_points
477   Haaland      FWD    13   14.9           104
4     Gabriel      DEF     1    6.4            81
135   Semenyo      MID     4    7.9            75
279     Muñoz      DEF     8    5.9            74
7    J.Timber      DEF     1    6.3            72
19       Rice      MID     1    6.9            71
283     Guéhi      DEF     8    5.1            69
280   Lacroix      DEF     8    5.0            67
322     Keane      DEF     9    4.5            65
194    Thiago      FWD     5    6.5            64


In [34]:
# 5_collect_historical_data.py
import time

def collect_all_historical_data():
    players = pd.read_csv('Data/current_players.csv')
    
    print(f"Collecting history for {len(players)} players...")
    
    all_history = []
    failed_count = 0
    
    for i, player_id in enumerate(players['id']):
        if (i + 1) % 100 == 0:
            print(f"Progress: {i+1}/{len(players)}")
        
        try:
            history = collector.get_player_history(player_id)
            if not history.empty:
                history['player_id'] = player_id
                all_history.append(history)
            time.sleep(0.1)  # Be nice to API
        except:
            failed_count += 1
            continue
    
    print(f"\n✅ Collected {len(all_history)} player histories")
    print(f"⚠️  Failed: {failed_count} players")
    
    if all_history:
        full_history = pd.concat(all_history, ignore_index=True)
        full_history.to_csv('Data/player_history.csv', index=False)
        print(f"✅ Saved to Data/player_history.csv")
        return full_history
    
    return pd.DataFrame()

# Collect history
history = collect_all_historical_data()

Progress: 100/755
Progress: 200/755
Progress: 300/755
Progress: 400/755
Progress: 500/755
Progress: 600/755
Progress: 700/755

✅ Collected 755 player histories
⚠️  Failed: 0 players
✅ Saved to Data/player_history.csv


Found 755 players
Progress: 50/755 players processed
Progress: 100/755 players processed
Progress: 150/755 players processed
Progress: 200/755 players processed
Progress: 250/755 players processed
Progress: 300/755 players processed
Progress: 350/755 players processed
Progress: 400/755 players processed
Progress: 450/755 players processed
Progress: 500/755 players processed
Progress: 550/755 players processed
Progress: 600/755 players processed
Progress: 650/755 players processed
Progress: 700/755 players processed
Progress: 750/755 players processed

Collected 755 player histories
Failed for 0 players
✅ Saved 8818 records to Data/player_history.csv
